# Step 1: Build PyG Graph Data

In [1]:
import numpy as np
import pickle
import torch
from torch_geometric.data import Data

def load_variable(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

def save_variable(v, filename):
    with open(filename, 'wb') as f:
        pickle.dump(v, f)

PROC_PATH = "/Users/psyche910/Desktop/Master.1/thesis/paper code/02 data processing"
SAVE_PATH = "/Users/psyche910/Desktop/Master.1/thesis/paper code/03 model"

/Users/psyche910/opt/anaconda3/envs/thesis/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## Clean Data Load

In [2]:
cities = ['ANTWERP', 'BANGKOK', 'BARCELONA']

data_dict = {}
for city in cities:
    nodes = np.array(load_variable(f"{PROC_PATH}/{city}_nodes_clean.txt"))  # (N, 361, 4)
    edges = np.array(load_variable(f"{PROC_PATH}/{city}_edges_clean.txt"))  # (2, E)
    print(f"{city}: nodes={nodes.shape}, edges={edges.shape}")
    data_dict[city] = {'nodes': nodes, 'edges': edges}

ANTWERP: nodes=(21899, 361, 4), edges=(2, 65169)
BANGKOK: nodes=(26018, 361, 4), edges=(2, 85414)
BARCELONA: nodes=(15638, 361, 4), edges=(2, 54604)


## Build PyG Data 

pre disaster data （x）

In [3]:
def build_pyg_data(nodes, edges):
    """
    nodes: (N, 361, 4)
    edges: (2, E)
    返回 PyG Data，x shape = (N, 180, 4) 只用灾前
    """
    x_pre    = nodes[:, :180, :].astype(np.float32)   # (N, 180, 4) 灾前
    x_during = nodes[:, 180:, :].astype(np.float32)   # (N, 181, 4) 灾后
    edges    = edges.astype(np.int64)                  # uint64 → int64

    x_pre_tensor    = torch.tensor(x_pre,    dtype=torch.float32)
    x_during_tensor = torch.tensor(x_during, dtype=torch.float32)
    edge_index      = torch.tensor(edges,    dtype=torch.long)

    data = Data(
        x=x_pre_tensor,           # (N, 180, 4) 灾前特征，GAT+Transformer 的输入
        y=x_during_tensor,        # (N, 181, 4) 灾后标签，预测目标
        edge_index=edge_index,    # (2, E)
    )
    return data

pyg_data = {}
for city in cities:
    pyg_data[city] = build_pyg_data(data_dict[city]['nodes'], data_dict[city]['edges'])
    d = pyg_data[city]
    print(f"{city}: x={d.x.shape}, y={d.y.shape}, edge_index={d.edge_index.shape}")

ANTWERP: x=torch.Size([21899, 180, 4]), y=torch.Size([21899, 181, 4]), edge_index=torch.Size([2, 65169])
BANGKOK: x=torch.Size([26018, 180, 4]), y=torch.Size([26018, 181, 4]), edge_index=torch.Size([2, 85414])
BARCELONA: x=torch.Size([15638, 180, 4]), y=torch.Size([15638, 181, 4]), edge_index=torch.Size([2, 54604])


## Normalization

Perform min-max normalization on the pre-disaster data for each node, using the pre-disaster min and max values, and apply the same scaling to the during-disaster data.

In [4]:
def normalize(pyg_data):
    x = pyg_data.x  # (N, 180, 4)
    y = pyg_data.y  # (N, 181, 4)

    # Global min-max using pre-event data only
    x_min = x.min()
    x_max = x.max()
    denom = (x_max - x_min).clamp(min=1e-8)

    pyg_data.x = (x - x_min) / denom
    pyg_data.y = (y - x_min) / denom   # same scale applied to during-event
    pyg_data.x_min = x_min             # save for inverse transform
    pyg_data.x_max = x_max
    return pyg_data

for city in cities:
    pyg_data[city] = normalize(pyg_data[city])
    print(f"{city}: x range [{pyg_data[city].x.min():.3f}, {pyg_data[city].x.max():.3f}]")

ANTWERP: x range [0.000, 1.000]
BANGKOK: x range [0.000, 1.000]
BARCELONA: x range [0.000, 1.000]


In [5]:
for city in cities:
    torch.save(pyg_data[city], f"{SAVE_PATH}/{city}_pyg.pt")
    print(f"{city} saved.")

ANTWERP saved.
BANGKOK saved.
BARCELONA saved.
